# 06. Registering a Function App MCP Server as a Foundry Project Connection

**Difficulty: Advanced**

This notebook annotates `connect_foundry_mcp.py`, which registers the C# Azure Function App in this folder (`GetOrderStatus.cs` / `ListCustomerOrders.cs`, exposed over MCP by the Functions runtime) as a **project connection** in an Azure AI Foundry project — by calling the **Azure Resource Manager (ARM) REST API** directly with `requests`, not through any Python SDK.

A project *connection* stores the MCP server's URL and its function key once, centrally, inside the Foundry project. After this runs, the MCP server shows up under the project's **Connected resources** in the Foundry portal, and agents built in that project can reference the connection by name instead of every script re-supplying the URL + key (the way the sibling notebook `agent_with_functionapp_mcp.ipynb` does inline). Chapter 08's `02_project_mcp_conn.py` uses this exact same pattern to register a *different* MCP server — comparing the two shows what's generic (the ARM call) vs. per-server (target URL, key, connection name).

## Prerequisites

**pip3 packages** (all already in the repo root `requirements.txt`):
```bash
pip3 install requests azure-identity
```

**Azure resources required:**
- An Azure AI Foundry **project** you have Contributor (or higher) rights on — the connection is written into the project's ARM resource, so this is a *management-plane* operation.
- The demo **Azure Function App** in this folder deployed to Azure with its MCP extension enabled, plus its **system key** for the `mcp_extension` webhook (Portal → Function App → App keys → System keys).

**Auth:** `az login` — `DefaultAzureCredential` picks up your Azure CLI session. No API key is used for the ARM call itself; only the *stored* credential (the function key) is an API key.

**Env vars read via [`azure_config.py`](../../azure_config.py)** (add to the repo-root `.env`):
- `AZURE_AI_PROJECT_RESOURCE_ID` — the project's full ARM resource ID (`/subscriptions/.../resourceGroups/.../providers/Microsoft.CognitiveServices/accounts/<account>/projects/<project>`)
- `AZURE_FUNCTION_APP_HOST` — Function App hostname, no scheme (e.g. `my-func.azurewebsites.net`)
- `MCP_SYSTEM_KEY` — the Function App's MCP system key
- `AZURE_AI_PROJECT_MCP_CONNECTION_NAME` — optional; this script defaults it to `soubhik-demo-funcapp-mcp-connection`

## What You'll Learn

- The difference between Azure's **management plane** (ARM, `https://management.azure.com`) and **data plane** (the project endpoint your agents call) — and why this script needs an ARM-scoped token
- How `get_bearer_token_provider` turns a credential into a callable that mints tokens for a given scope
- The exact JSON shape of a Foundry project **connection** resource: `authType: CustomKeys`, `category: RemoteTool`, a `target` URL, stored `credentials`, and `metadata` declaring the MCP transport
- Why `PUT` (rather than `POST`) makes the registration **idempotent** — safe to re-run
- Where this connection then appears (Foundry portal → project → Connected resources) and who can use it (`isSharedToAll`)

### Step 1 — Repo-root config bootstrap and imports

Like every script in this chapter, configuration comes from the repo-root [`azure_config.py`](../../azure_config.py) `config` object rather than per-file `os.getenv` calls. The `for` loop walks up from the current directory until it finds the folder containing `azure_config.py` and puts it on `sys.path` (in a notebook there is no `__file__`, so it starts from `Path.cwd()` — which works because this notebook lives inside the chapter).

`config.project_mcp_connection_name(...)` is a **method with a per-script default**, not a property: Chapter 08 registers a different MCP connection in the same project, and a single shared default name would make the two scripts silently overwrite each other's connection.

💡 **Exam tip:** AI-102/AI-103 expects you to know that connections (to Azure OpenAI, AI Search, storage, custom/remote tools…) are defined **at the project (or hub) level** and shared by the assets inside it — that centralization is the whole point of a Foundry project.

In [ ]:
import sys
from pathlib import Path

_start = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
for _parent in [_start, *_start.parents]:
    if (_parent / "azure_config.py").exists():
        sys.path.insert(0, str(_parent))
        break

from azure_config import config

import requests
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

credential = DefaultAzureCredential()

PROJECT_RESOURCE_ID = config.project_resource_id

PROJECT_CONNECTION_NAME = config.project_mcp_connection_name("soubhik-demo-funcapp-mcp-connection")

FUNCTION_APP_HOST = config.function_app_host
MCP_SYSTEM_KEY = config.mcp_system_key

mcp_endpoint = f"https://{FUNCTION_APP_HOST}/runtime/webhooks/mcp/sse"

### Step 2 — An ARM-scoped bearer token

Everything else in this chapter authenticates against the **data plane** (the project endpoint, scope `https://ai.azure.com/.default` or an OpenAI-compatible endpoint). Creating a *connection*, however, means writing an Azure **resource** — so the token must be scoped to ARM: `https://management.azure.com/.default`.

`get_bearer_token_provider(credential, scope)` returns a zero-argument callable; each call yields a valid (cached, auto-refreshed) token string. Here it's called once to build a plain `Authorization: Bearer …` header for `requests`.

💡 **Exam tip:** know the two planes and their scopes. Management plane = ARM (`management.azure.com`) — create/configure resources, RBAC roles like *Contributor*. Data plane = the service endpoint — call models, run agents, roles like *Cognitive Services User*. A principal can have rights on one plane and none on the other.

🔄 **Alternatives:** `az rest --method put --url ...` does the same call with the CLI's own token; the Azure SDK's `azure-mgmt-cognitiveservices` package could do it with typed models; or click it together in the Foundry portal under **Connected resources → New connection**. The raw REST call is shown because the `2025-10-01-preview` connection shape for remote MCP tools landed in the REST API before the SDKs/portal caught up.

In [ ]:
bearer_token_provider = get_bearer_token_provider(
    credential,
    "https://management.azure.com/.default"
)

headers = {
    "Authorization": f"Bearer {bearer_token_provider()}",
    "Content-Type": "application/json"
}

### Step 3 — `PUT` the connection resource

The URL addresses the connection as a **child resource of the project**: `https://management.azure.com{project_resource_id}/connections/{name}?api-version=2025-10-01-preview`. `PUT` on a full resource path is create-*or*-update — running this twice is safe; the second run just overwrites the same connection (idempotent, unlike `POST`).

The payload is the connection's ARM shape:

- `authType: "CustomKeys"` — the stored credential is one or more named header keys; here the single `x-functions-key` header the Functions MCP webhook requires.
- `category: "RemoteTool"` — marks this connection as a remote tool endpoint (an MCP server), as opposed to e.g. `AzureOpenAI` or `CognitiveSearch` connections.
- `target` — the tool's URL: the Function App's MCP SSE webhook.
- `isSharedToAll: true` — every user of the project can use the connection (without being able to read the raw key back).
- `metadata.McpTransport: "sse"` — tells Foundry which MCP transport the server speaks.

💡 **Exam tip:** stored connection credentials are **write-only** — portal users and agents *use* the connection, but the key is never displayed again. That secrets-centralization story (vs. keys pasted into every script) is a favorite exam angle, alongside Key Vault.

🔄 **Alternatives:** for OAuth-protected MCP servers the connection would use a different `authType`; for a server with no auth at all you could skip the connection entirely and pass the URL inline per request — which is exactly what the sibling `agent_with_functionapp_mcp.ipynb` does with its `headers` field.

In [ ]:
response = requests.put(
    f"https://management.azure.com{PROJECT_RESOURCE_ID}/connections/{PROJECT_CONNECTION_NAME}"
    "?api-version=2025-10-01-preview",
    headers=headers,
    json={
        "name": PROJECT_CONNECTION_NAME,
        "type": "Microsoft.MachineLearningServices/workspaces/connections",
        "properties": {
            "authType": "CustomKeys",
            "category": "RemoteTool",
            "target": mcp_endpoint,
            "isSharedToAll": True,
            "credentials": {
                "keys": {
                    "x-functions-key": MCP_SYSTEM_KEY
                }
            },
            "metadata": {
                "ApiType": "Azure",
                "McpTransport": "sse"
            }
        }
    }
)

print(response.status_code)
print(response.text)

response.raise_for_status()

print(f"Connection '{PROJECT_CONNECTION_NAME}' created or updated successfully.")

### Step 4 — Verifying the result

A `200` (updated) or `201` (created) comes back with the connection resource echoed as JSON — minus the credentials, which ARM never returns. To see it:

- **Portal:** Foundry portal → your project → **Management center** → **Connected resources** — the connection appears with category *Remote Tool*.
- **CLI:** `az rest --method get --url "https://management.azure.com<project-resource-id>/connections?api-version=2025-10-01-preview"` lists every connection in the project.

From here, an agent defined in this project can reference the MCP server by **connection name** and Foundry injects the stored URL + key at run time.

## Summary

This notebook annotated `connect_foundry_mcp.py`, which registers this folder's Function App MCP server as a reusable **project connection** in Azure AI Foundry with one idempotent `PUT` to the ARM REST API. The key ideas: management-plane vs. data-plane tokens (`management.azure.com/.default` here), the connection resource shape (`CustomKeys` + `RemoteTool` + `target` + `McpTransport: sse`), and why centralizing the URL + function key in the project beats pasting them into every agent script. The sibling notebook `agent_with_functionapp_mcp.ipynb` shows the opposite, connection-less approach — passing the same URL and key inline on a single Responses API call.

## Try It Yourself

1. **Easy:** Re-run the cell and confirm the second run returns `200` instead of `201` — the visible proof that `PUT` is create-or-update.
2. **Intermediate:** List the project's connections with `az rest` (Step 4) and find your connection in the JSON; confirm the `credentials` are *not* in the response.
3. **Advanced:** Register a *second* connection pointing at the local `func start` endpoint of `03-cloudxeus-mcp` exposed through a dev tunnel (`az webapp` dev tunnels or ngrok), then delete it again with `--method delete` — a full connection lifecycle without touching the portal.